# LoRA (Low-Rank Adaptation) Implementation with PyTorch

## What is LoRA?
    LoRA is a technique to fine-tune large neural networks efficiently by
    adding small trainable matrices (A and B) to existing weight matrices,
    instead of training all parameters.

## Key Idea:
    Instead of updating W (large), we keep W frozen and add a low-rank 
    decomposition: ΔW = B × A (where B and A are much smaller matrices)

## Visual representation:

      Original:     W (d × k)  [millions of parameters]
      
      LoRA:         W + B × A
                    where B is (d × r) and A is (r × k)
                    and r << min(d, k)  [thousands of parameters]

In [1]:
import torch
import torchvision.datasets as datasets 
import torchvision.transforms as transforms
import torch.nn as nn
import matplotlib.pyplot as plt
from tqdm import tqdm

## SETUP: Make experiments reproducible

In [2]:
_ = torch.manual_seed(0)

## DATA PREPARATION: MNIST Dataset

In [3]:
# MNIST: 28x28 grayscale images of handwritten digits (0-9)
# We'll train a network to classify these digits, then fine-tune it on digit 9

# Normalize with MNIST mean (0.1307) and std (0.3081)
transform = transforms.Compose([
    transforms.ToTensor(), 
    transforms.Normalize((0.1307,), (0.3081,))
])

# Load training data
mnist_trainset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
train_loader = torch.utils.data.DataLoader(mnist_trainset, batch_size=10, shuffle=True)

# Load test data
mnist_testset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)
test_loader = torch.utils.data.DataLoader(mnist_testset, batch_size=10, shuffle=True)

# Use GPU if available, otherwise CPU
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

## MODEL DEFINITION: An intentionally large network

We create an "oversized" network to better demonstrate LoRA's efficiency

In practice, LoRA shines with truly large models (like GPT, BERT, etc.)

Network Architecture:
    
    Input (784) → Linear1 (1000) → ReLU → Linear2 (2000) → ReLU → Linear3 (10)

Visual:
  
    [28×28 image] → [flatten to 784] → [1000 neurons] → [2000 neurons] → [10 classes]
                   784,000 params      2,000,000 params    20,000 params

In [4]:
class RichBoyNet(nn.Module):
    """
    An overly large neural network for MNIST classification.
    Total parameters: ~2.8 million (way more than needed for MNIST!)
    """
    def __init__(self, hidden_size_1=1000, hidden_size_2=2000):
        super(RichBoyNet, self).__init__()
        # Layer 1: 784 input features → 1000 hidden units
        self.linear1 = nn.Linear(28*28, hidden_size_1) 
        # Layer 2: 1000 hidden units → 2000 hidden units
        self.linear2 = nn.Linear(hidden_size_1, hidden_size_2) 
        # Layer 3: 2000 hidden units → 10 output classes
        self.linear3 = nn.Linear(hidden_size_2, 10)
        self.relu = nn.ReLU()

    def forward(self, img):
        # Flatten image from [batch, 1, 28, 28] to [batch, 784]
        x = img.view(-1, 28*28)
        # Forward pass through layers with ReLU activations
        x = self.relu(self.linear1(x))
        x = self.relu(self.linear2(x))
        x = self.linear3(x)  # No activation on output (using CrossEntropy loss)
        return x

net = RichBoyNet().to(device)

## TRAINING FUNCTION: Standard supervised learning

In [5]:
def train(train_loader, net, epochs=5, total_iterations_limit=None):
    """
    Train the network using Cross-Entropy loss and Adam optimizer.
    
    Args:
        train_loader: DataLoader with training data
        net: Neural network to train
        epochs: Number of passes through the dataset
        total_iterations_limit: Optional limit on training iterations
    """
    cross_el = nn.CrossEntropyLoss()  # Standard loss for classification
    optimizer = torch.optim.Adam(net.parameters(), lr=0.001)

    total_iterations = 0

    for epoch in range(epochs):
        net.train()  # Set network to training mode
        loss_sum = 0
        num_iterations = 0

        # Progress bar for visual feedback
        data_iterator = tqdm(train_loader, desc=f'Epoch {epoch+1}')
        if total_iterations_limit is not None:
            data_iterator.total = total_iterations_limit
            
        for data in data_iterator:
            num_iterations += 1
            total_iterations += 1
            x, y = data
            x = x.to(device)
            y = y.to(device)
            
            # Standard training loop
            optimizer.zero_grad()                    # Reset gradients
            output = net(x.view(-1, 28*28))         # Forward pass
            loss = cross_el(output, y)              # Compute loss
            loss_sum += loss.item()
            avg_loss = loss_sum / num_iterations
            data_iterator.set_postfix(loss=avg_loss)
            loss.backward()                          # Backward pass
            optimizer.step()                         # Update weights

            if total_iterations_limit is not None and total_iterations >= total_iterations_limit:
                return

## INITIAL TRAINING: Pre-train the network (simulating a base model)

In [6]:
print("Training base model on all MNIST digits...")
train(train_loader, net, epochs=1)

Training base model on all MNIST digits...


Epoch 1: 100%|████████████████████████████████████████████████████████| 6000/6000 [00:26<00:00, 224.21it/s, loss=0.234]


## CHECKPOINT: Save original weights

In [7]:
# We'll use this later to verify that LoRA doesn't modify original weights
original_weights = {}
for name, param in net.named_parameters():
    original_weights[name] = param.clone().detach()

In [8]:
original_weights

{'linear1.weight': tensor([[ 0.0025,  0.0219, -0.0266,  ...,  0.0247,  0.0065,  0.0048],
         [ 0.0204,  0.0252,  0.0297,  ...,  0.0199,  0.0342,  0.0102],
         [ 0.0020,  0.0371, -0.0111,  ...,  0.0018,  0.0233,  0.0302],
         ...,
         [ 0.0287,  0.0923,  0.0905,  ...,  0.0595,  0.0856,  0.0331],
         [ 0.0640,  0.0273,  0.0130,  ...,  0.0473,  0.0409,  0.0417],
         [ 0.0221, -0.0016,  0.0496,  ...,  0.0590,  0.0324,  0.0428]],
        device='cuda:0'),
 'linear1.bias': tensor([-2.7360e-02, -1.8806e-02, -2.9787e-02, -1.4327e-02, -1.5282e-02,
         -6.1148e-02, -3.4358e-02, -1.8904e-02, -4.7943e-02, -7.0964e-03,
         -2.2674e-02, -9.6570e-03, -2.7861e-02, -3.1330e-02, -3.6972e-03,
         -2.7190e-02, -5.2847e-02, -3.5449e-02, -2.6880e-02, -4.6640e-02,
         -3.2971e-02, -6.4643e-02,  2.0789e-02, -5.6361e-02, -1.1986e-02,
         -3.9625e-02, -3.8167e-02, -1.2098e-02,  5.3412e-05, -1.8758e-02,
          2.4631e-03, -3.6834e-02, -1.9359e-02, -3.8924

## EVALUATION FUNCTION: Test network accuracy

In [9]:
def test():
    """
    Evaluate the network on the test set and report accuracy per digit.
    This helps us identify which digits the model struggles with.
    """
    correct = 0
    total = 0
    wrong_counts = [0 for i in range(10)]  # Track errors per digit

    with torch.no_grad():  # No gradients needed for testing
        for data in tqdm(test_loader, desc='Testing'):
            x, y = data
            x = x.to(device)
            y = y.to(device)
            output = net(x.view(-1, 784))
            
            # Check each prediction
            for idx, i in enumerate(output):
                if torch.argmax(i) == y[idx]:
                    correct += 1
                else:
                    wrong_counts[y[idx]] += 1
                total += 1
                
    print(f'Accuracy: {round(correct/total, 3)}')
    for i in range(len(wrong_counts)):
        print(f'wrong counts for the digit {i}: {wrong_counts[i]}')

print("\nTesting base model performance:")
test()


Testing base model performance:


Testing: 100%|████████████████████████████████████████████████████████████████████| 1000/1000 [00:03<00:00, 324.30it/s]

Accuracy: 0.957
wrong counts for the digit 0: 16
wrong counts for the digit 1: 8
wrong counts for the digit 2: 65
wrong counts for the digit 3: 87
wrong counts for the digit 4: 13
wrong counts for the digit 5: 13
wrong counts for the digit 6: 39
wrong counts for the digit 7: 57
wrong counts for the digit 8: 32
wrong counts for the digit 9: 98


## PARAMETER ANALYSIS: Count parameters in original network

In [10]:
print("\n" + "="*70)
print("ORIGINAL NETWORK PARAMETERS")
print("="*70)
total_parameters_original = 0
for index, layer in enumerate([net.linear1, net.linear2, net.linear3]):
    total_parameters_original += layer.weight.nelement() + layer.bias.nelement()
    print(f'Layer {index+1}: W: {layer.weight.shape} + B: {layer.bias.shape}')
print(f'Total number of parameters: {total_parameters_original:,}')


ORIGINAL NETWORK PARAMETERS
Layer 1: W: torch.Size([1000, 784]) + B: torch.Size([1000])
Layer 2: W: torch.Size([2000, 1000]) + B: torch.Size([2000])
Layer 3: W: torch.Size([10, 2000]) + B: torch.Size([10])
Total number of parameters: 2,807,010


## LoRA IMPLEMENTATION: The magic happens here!

LoRA Mathematical Formulation:

    h = W₀x + ΔWx = W₀x + BAx

Where:
  - W₀: Original pretrained weights (frozen)
  - ΔW = BA: Low-rank update
  - B: Matrix of size (d × r)
  - A: Matrix of size (r × k)
  - r: Rank (r << min(d,k))

Visual representation of dimensions:

    Original Weight Matrix W₀:
    ┌─────────────────────┐
    │                     │  d (e.g., 1000)
    │         W₀          │
    │                     │
    └─────────────────────┘
         k (e.g., 784)
    
    LoRA Decomposition:
    ┌─────┐   ┌─────────────────────┐
    │     │   │          A          │  d
    │  B  │ × └─────────────────────┘  r (e.g., 1)
    │     │             r×k
    └─────┘   
    d×r              

Result: d×k matrix (same size as W₀)

Parameters: d×r + r×k (much smaller than d×k when r is small!)

In [11]:
class LoRAParametrization(nn.Module):
    """
    LoRA parameterization module that adds low-rank adaptation to a weight matrix.
    
    This implements the core LoRA equation: W_new = W_original + (B @ A) * scale
    
    Args:
        features_in: Input dimension (d)
        features_out: Output dimension (k)
        rank: Rank of the low-rank decomposition (r)
        alpha: Scaling factor (usually set to rank)
        device: Device to place tensors on
    """
    def __init__(self, features_in, features_out, rank=1, alpha=1, device='cpu'):
        super().__init__()
        
        # Initialize LoRA matrices A and B

        # From the paper (Section 4.1):
        # "We use a random Gaussian initialization for A and zero for B,
        #  so ΔW = BA is zero at the beginning of training"
        #
        # This ensures the model starts with the same behavior as the
        # pretrained model (since ΔW = 0 initially)

        
        # Matrix A: (r × k) - initialized with Gaussian noise
        self.lora_A = nn.Parameter(torch.zeros((rank, features_out)).to(device))
        # Matrix B: (d × r) - initialized to zero
        self.lora_B = nn.Parameter(torch.zeros((features_in, rank)).to(device))
        nn.init.normal_(self.lora_A, mean=0, std=1)
        

        # Scaling factor α/r

        # From the paper (Section 4.1):
        # "We then scale ΔWx by α/r, where α is a constant in r.
        #  When optimizing with Adam, tuning α is roughly the same as tuning
        #  the learning rate if we scale the initialization appropriately.
        #  As a result, we simply set α to the first r we try and do not tune it."
        #
        # This scaling helps stabilize training across different rank values

        self.scale = alpha / rank
        self.enabled = True

    def forward(self, original_weights):
        """
        Apply LoRA adaptation to original weights.
        
        Computation:
            W_effective = W_original + (B @ A) * scale
        
        Visual:
            ┌──────────┐   ┌─────┐   ┌─────┐
            │    W₀    │ + │  B  │ @ │  A  │ × (α/r)
            └──────────┘   └─────┘   └─────┘
               d × k        d × r     r × k
        
        Returns:
            Effective weight matrix (same size as original)
        """
        if self.enabled:
            # Compute ΔW = BA and add to original weights
            # torch.matmul(B, A) gives us the low-rank update
            # .view() reshapes to match original weight dimensions
            return original_weights + torch.matmul(self.lora_B, self.lora_A).view(original_weights.shape) * self.scale
        else:
            # When disabled, return original weights unchanged
            return original_weights

## APPLY LoRA: Add parameterization to network layers

In [12]:
import torch.nn.utils.parametrize as parametrize

def linear_layer_parameterization(layer, device, rank=1, lora_alpha=1):
    """
    Create a LoRA parameterization for a linear layer.
    
    Note: We only adapt weight matrices, not biases.
    
    From the paper (Section 4.2):
    "We limit our study to only adapting the attention weights for downstream
     tasks and freeze the MLP modules (so they are not trained in downstream
     tasks) both for simplicity and parameter-efficiency.
     We leave the empirical investigation of [...] and biases to a future work."
    """
    features_in, features_out = layer.weight.shape
    return LoRAParametrization(
        features_in, features_out, rank=rank, alpha=lora_alpha, device=device
    )

# Register LoRA parameterization for each linear layer
# This modifies how the 'weight' attribute works in each layer
print("\n" + "="*70)
print("APPLYING LoRA PARAMETERIZATION")
print("="*70)

parametrize.register_parametrization(
    net.linear1, "weight", linear_layer_parameterization(net.linear1, device)
)
parametrize.register_parametrization(
    net.linear2, "weight", linear_layer_parameterization(net.linear2, device)
)
parametrize.register_parametrization(
    net.linear3, "weight", linear_layer_parameterization(net.linear3, device)
)

def enable_disable_lora(enabled=True):
    """
    Toggle LoRA on/off for all layers.
    When disabled, the network behaves exactly like the original pretrained model.
    """
    for layer in [net.linear1, net.linear2, net.linear3]:
        layer.parametrizations["weight"][0].enabled = enabled


APPLYING LoRA PARAMETERIZATION


## PARAMETER COMPARISON: Original vs LoRA-augmented

In [13]:
print("\n" + "="*70)
print("PARAMETER COMPARISON")
print("="*70)

total_parameters_lora = 0
total_parameters_non_lora = 0

for index, layer in enumerate([net.linear1, net.linear2, net.linear3]):
    lora_params = (layer.parametrizations["weight"][0].lora_A.nelement() + 
                   layer.parametrizations["weight"][0].lora_B.nelement())
    total_parameters_lora += lora_params
    total_parameters_non_lora += layer.weight.nelement() + layer.bias.nelement()
    
    print(f'\nLayer {index+1}:')
    print(f'  Original: W: {layer.weight.shape} + B: {layer.bias.shape}')
    print(f'  LoRA:     A: {layer.parametrizations["weight"][0].lora_A.shape} + '
          f'B: {layer.parametrizations["weight"][0].lora_B.shape}')
    print(f'  LoRA params: {lora_params:,}')

# Verify our parameter count matches the original
assert total_parameters_non_lora == total_parameters_original

print(f'\n{"="*70}')
print(f'Total parameters (original):           {total_parameters_non_lora:,}')
print(f'Total parameters (original + LoRA):    {total_parameters_lora + total_parameters_non_lora:,}')
print(f'Parameters introduced by LoRA:         {total_parameters_lora:,}')
parameters_increment = (total_parameters_lora / total_parameters_non_lora) * 100
print(f'Parameters increment:                  {parameters_increment:.3f}%')
print(f'{"="*70}')


PARAMETER COMPARISON

Layer 1:
  Original: W: torch.Size([1000, 784]) + B: torch.Size([1000])
  LoRA:     A: torch.Size([1, 784]) + B: torch.Size([1000, 1])
  LoRA params: 1,784

Layer 2:
  Original: W: torch.Size([2000, 1000]) + B: torch.Size([2000])
  LoRA:     A: torch.Size([1, 1000]) + B: torch.Size([2000, 1])
  LoRA params: 3,000

Layer 3:
  Original: W: torch.Size([10, 2000]) + B: torch.Size([10])
  LoRA:     A: torch.Size([1, 2000]) + B: torch.Size([10, 1])
  LoRA params: 2,010

Total parameters (original):           2,807,010
Total parameters (original + LoRA):    2,813,804
Parameters introduced by LoRA:         6,794
Parameters increment:                  0.242%


## FINE-TUNING WITH LoRA: Train only on digit 9

Strategy:
1. Freeze all original weights (W₀)
2. Train only LoRA parameters (A and B matrices)
3. Use only digit 9 samples
4. Train for just 100 batches

Goal: Improve performance on digit 9 without forgetting other digits

In [14]:
print("\n" + "="*70)
print("FINE-TUNING WITH LoRA")
print("="*70)

# Freeze all non-LoRA parameters
for name, param in net.named_parameters():
    if 'lora' not in name:
        print(f'Freezing: {name}')
        param.requires_grad = False

# Prepare dataset with only digit 9
mnist_trainset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
exclude_indices = mnist_trainset.targets == 9  # Keep only 9s
mnist_trainset.data = mnist_trainset.data[exclude_indices]
mnist_trainset.targets = mnist_trainset.targets[exclude_indices]
train_loader = torch.utils.data.DataLoader(mnist_trainset, batch_size=10, shuffle=True)

print(f'\nFine-tuning on {len(mnist_trainset)} samples of digit 9...')
print(f'Training for 100 batches only...\n')

# Fine-tune for just 100 iterations
train(train_loader, net, epochs=1, total_iterations_limit=100)


FINE-TUNING WITH LoRA
Freezing: linear1.bias
Freezing: linear1.parametrizations.weight.original
Freezing: linear2.bias
Freezing: linear2.parametrizations.weight.original
Freezing: linear3.bias
Freezing: linear3.parametrizations.weight.original

Fine-tuning on 5949 samples of digit 9...
Training for 100 batches only...



Epoch 1:  99%|███████████████████████████████████████████████████████████▍| 99/100 [00:00<00:00, 131.73it/s, loss=0.11]


## VERIFICATION: Ensure original weights are unchanged

In [17]:
print("\n" + "="*70)
print("VERIFYING WEIGHT INTEGRITY")
print("="*70)

# Check that frozen parameters remain unchanged
assert torch.all(net.linear1.parametrizations.weight.original == original_weights['linear1.weight'])
assert torch.all(net.linear2.parametrizations.weight.original == original_weights['linear2.weight'])
assert torch.all(net.linear3.parametrizations.weight.original == original_weights['linear3.weight'])
print("Original weights unchanged!")

# Verify the LoRA equation: W_effective = W_original + BA * scale
enable_disable_lora(enabled=True)
assert torch.equal(
    net.linear1.weight, 
    net.linear1.parametrizations.weight.original + 
    (net.linear1.parametrizations.weight[0].lora_B @ 
     net.linear1.parametrizations.weight[0].lora_A) * 
    net.linear1.parametrizations.weight[0].scale
)
print("LoRA equation verified!")

# Verify that disabling LoRA restores original behavior
enable_disable_lora(enabled=False)
assert torch.equal(net.linear1.weight, original_weights['linear1.weight'])
print("LoRA disable functionality works!")


VERIFYING WEIGHT INTEGRITY
Original weights unchanged!
LoRA equation verified!
LoRA disable functionality works!


## FINAL EVALUATION: Compare performance with/without LoRA

In [18]:
print("\n" + "="*70)
print("TESTING WITH LoRA ENABLED")
print("="*70)
print("Expected: Better performance on digit 9, possible degradation on others")
enable_disable_lora(enabled=True)
test()

print("\n" + "="*70)
print("TESTING WITH LoRA DISABLED")
print("="*70)
print("Expected: Same performance as original pretrained model")
enable_disable_lora(enabled=False)
test()


TESTING WITH LoRA ENABLED
Expected: Better performance on digit 9, possible degradation on others


Testing: 100%|████████████████████████████████████████████████████████████████████| 1000/1000 [00:03<00:00, 252.10it/s]


Accuracy: 0.843
wrong counts for the digit 0: 16
wrong counts for the digit 1: 12
wrong counts for the digit 2: 87
wrong counts for the digit 3: 197
wrong counts for the digit 4: 727
wrong counts for the digit 5: 82
wrong counts for the digit 6: 46
wrong counts for the digit 7: 138
wrong counts for the digit 8: 254
wrong counts for the digit 9: 16

TESTING WITH LoRA DISABLED
Expected: Same performance as original pretrained model


Testing: 100%|████████████████████████████████████████████████████████████████████| 1000/1000 [00:03<00:00, 302.82it/s]

Accuracy: 0.957
wrong counts for the digit 0: 16
wrong counts for the digit 1: 8
wrong counts for the digit 2: 65
wrong counts for the digit 3: 87
wrong counts for the digit 4: 13
wrong counts for the digit 5: 13
wrong counts for the digit 6: 39
wrong counts for the digit 7: 57
wrong counts for the digit 8: 32
wrong counts for the digit 9: 98
